# 06 — Batch inference and prediction logging

Production predictions are logged with timestamps and the active model version. Avoid storing personally identifiable or sensitive input data in real deployments.

In [1]:
import json, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

import joblib
from datetime import datetime, timezone
from pathlib import Path
model_path = Path('../artifacts/model_v1.joblib')
if model_path.exists(): model = joblib.load(model_path)
else:
    model = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]).fit(X_train, y_train)
batch = X_test.iloc[:25]
prob = model.predict_proba(batch)[:,1]
logs = pd.DataFrame({'timestamp':datetime.now(timezone.utc).isoformat(),'model_version':'1.0.0','prediction':model.predict(batch),'positive_class_probability':prob})
logs.to_csv('../artifacts/prediction_log_sample.csv', index=False)
logs.head()

,timestamp,model_version,prediction,positive_class_probability
0,2026-10-11T05:43:25.186198+00:00,1.0.0,0,5.888242e-08
1,2026-10-11T05:43:25.186198+00:00,1.0.0,1,9.999887e-01
2,2026-10-11T05:43:25.186198+00:00,1.0.0,0,6.410825e-03
3,2026-10-11T05:43:25.186198+00:00,1.0.0,1,5.335085e-01
4,2026-10-11T05:43:25.186198+00:00,1.0.0,0,6.525001e-10
